# Lesson 3 Demo 4: Using the WHERE Clause
<img src="images/cassandralogo.png" width="250" height="250">

### In this exercise we are going to walk through the basics of using the WHERE clause in Apache Cassandra.

##### denotes where the code needs to be completed.

#### We will use a python wrapper/ python driver called cassandra to run the Apache Cassandra queries. This library should be preinstalled but in the future to install this library you can run this command in a notebook to install locally: 
! pip install cassandra-driver
#### More documentation can be found here: https://github.com/apache/cassandra-python-driver

#### Import Apache Cassandra python package

In [7]:
import cassandra
#!pip install cassandra-driver
print(cassandra.__version__)

3.30.0


### First let's create a connection to the database

In [8]:

from cassandra.cluster import Cluster
try: 
    cluster = Cluster(['127.0.0.1']) #If you have a locally installed Apache Cassandra instance
    session = cluster.connect()
except Exception as e:
    print(e)

DependencyException: Unable to load a default connection class
The following exceptions were observed: 
 - The C extension needed to use libev was not found.  This probably means that you didn't have the required build dependencies when installing the driver.  See https://docs.datastax.com/en/developer/python-driver/latest/installation/index.html#c-extensions for instructions on installing build dependencies and building the C extension.
 - Unable to import asyncore module.  Note that this module has been removed in Python 3.12 so when using the driver with this version (or anything newer) you will need to use one of the other event loop implementations.

### Let's create a keyspace to do our work in 

In [9]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS udacity 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)

except Exception as e:
    print(e)

name 'session' is not defined


#### Connect to our Keyspace. Compare this to how we had to create a new session in PostgreSQL.  

In [4]:
try:
    session.set_keyspace('udacity')
except Exception as e:
    print(e)

### Let's imagine we would like to start creating a new Music Library of albums. 
### We want to ask 4 question of our data
#### 1. Give me every album in my music library that was released in a 1965 year
`select * from music_library WHERE YEAR=1965`

#### 2. Give me the album that is in my music library that was released in 1965 by "The Beatles"
`select * from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles'`

#### 3. Give me all the albums released in a given year that was made in Oxford 
`select * from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles' AND ALBUM_NAME='Rubber Soul' AND CITY='Oxford'`

#### 4. Give me the city that the album "Rubber Soul" was recorded
`select city from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles' AND ALBUM_NAME='Rubber Soul'`

### Here is our Collection of Data
<img src="images/table3.png" width="650" height="350">

### How should we model this data? What should be our Primary Key and Partition Key? Since our data is looking for the YEAR let's start with that. From there we will add clustering columns on Artist Name and Album Name.

In [5]:
query = "CREATE TABLE IF NOT EXISTS music_library "
query = query + "(year int, artist_name text, album_name text, city text, PRIMARY KEY (year, artist_name, album_name, city))"
try:
    session.execute(query)
except Exception as e:
    print(e)

### Let's insert our data into of table

In [6]:
query = "INSERT INTO music_library (year, artist_name, album_name, city)"
query = query + " VALUES (%s, %s, %s, %s)"

try:
    session.execute(query, (1970, "The Beatles", "Let it Be", "Liverpool"))
except Exception as e:
    print(e)
    
try:
    session.execute(query, (1965, "The Beatles", "Rubber Soul", "Oxford"))
except Exception as e:
    print(e)
    
try:
    session.execute(query, (1965, "The Who", "My Generation", "London"))
except Exception as e:
    print(e)

try:
    session.execute(query, (1966, "The Monkees", "The Monkees", "Los Angeles"))
except Exception as e:
    print(e)

try:
    session.execute(query, (1970, "The Carpenters", "Close To You", "San Diego"))
except Exception as e:
    print(e)

### Let's Validate our Data Model with our 4 queries.

Query 1: 

In [7]:
query = "select * from music_library WHERE YEAR=1965"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

1965 The Beatles Rubber Soul Oxford
1965 The Who My Generation London


 Let's try the 2nd query.
 Query 2: 

In [8]:
query = "select * from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

1965 The Beatles Rubber Soul Oxford


### Let's try the 3rd query.
Query 3: 

In [9]:
query = "select * from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles' AND ALBUM_NAME='Rubber Soul' AND CITY='Oxford'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

1965 The Beatles Rubber Soul Oxford


In [10]:
query = "select city from music_library WHERE YEAR=1965 AND ARTIST_NAME='The Beatles' AND ALBUM_NAME='Rubber Soul'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.city)

Oxford


### And Finally close the session and cluster connection

In [11]:
session.shutdown()
cluster.shutdown()